# RADS Verification Notebook 01: Verify Metadata

This notebook loads `global_master_metadata.csv`, filters to the TUDAT dataset, inspects class distributions, generates a stratified 70/15/15 split, checks for data leakage, and logs the summary stats to W&B.


In [3]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Ensure project root is in path
project_root = Path("../..").resolve()
sys.path.insert(0, str(project_root))

from training.configs.config import load_training_config
from training.utils.logger import TrainingLogger
from training.utils.wandb_manager import TrainingWandbManager

# Load configuration
config = load_training_config()
print('Project root:', config.project_root)
print('Metadata path:', config.metadata_path)


Project root: E:\Rads
Metadata path: E:\Rads\Datasets\processed\global_master_metadata.csv


In [4]:
# Load metadata
df = pd.read_csv(config.metadata_path, low_memory=False)
print(f'Total rows in global metadata: {len(df)}')

# Filter to TUDAT
tudat_df = df[df['dataset_name'] == config.dataset_name].copy().reset_index(drop=True)
print(f'TUDAT rows: {len(tudat_df)}')


Total rows in global metadata: 5338
TUDAT rows: 111


In [5]:
# Inspect class distribution
label_col = config.label_column
class_counts = tudat_df[label_col].value_counts(dropna=False)
print('TUDAT Class Distribution:')
print(class_counts)


TUDAT Class Distribution:
type
non-accident    50
accident        44
challenging     17
Name: count, dtype: int64


In [6]:
# Stratified Split 70/15/15
from sklearn.model_selection import train_test_split

seed = config.seed
ratios = config.split_ratios
val_test_ratio = ratios['val'] + ratios['test']

# Filter out null labels
tudat_df = tudat_df[tudat_df[label_col].notna()].copy().reset_index(drop=True)
labels = tudat_df[label_col].astype(str)

# First split: train vs val_test
try:
    train_df, val_test_df = train_test_split(
        tudat_df,
        test_size=val_test_ratio,
        random_state=seed,
        stratify=labels
    )
except ValueError:
    print('Stratified split failed. Falling back to non-stratified split.')
    train_df, val_test_df = train_test_split(
        tudat_df,
        test_size=val_test_ratio,
        random_state=seed
    )

# Second split: val vs test
test_fraction = ratios['test'] / val_test_ratio
vt_labels = val_test_df[label_col].astype(str)
try:
    val_df, test_df = train_test_split(
        val_test_df,
        test_size=test_fraction,
        random_state=seed,
        stratify=vt_labels
    )
except ValueError:
    val_df, test_df = train_test_split(
        val_test_df,
        test_size=test_fraction,
        random_state=seed
    )

print(f'Train split size: {len(train_df)} ({len(train_df)/len(tudat_df):.1%})')
print(f'Val split size:   {len(val_df)} ({len(val_df)/len(tudat_df):.1%})')
print(f'Test split size:  {len(test_df)} ({len(test_df)/len(tudat_df):.1%})')


Train split size: 77 (69.4%)
Val split size:   17 (15.3%)
Test split size:  17 (15.3%)


In [7]:
# Verify class distributions in splits
print('Train distribution:')
print(train_df[label_col].value_counts())
print('\nVal distribution:')
print(val_df[label_col].value_counts())
print('\nTest distribution:')
print(test_df[label_col].value_counts())


Train distribution:
type
non-accident    35
accident        30
challenging     12
Name: count, dtype: int64

Val distribution:
type
non-accident    8
accident        7
challenging     2
Name: count, dtype: int64

Test distribution:
type
non-accident    7
accident        7
challenging     3
Name: count, dtype: int64


In [8]:
# Verify no data leakage (overlapping videos)
train_videos = set(train_df['original_path'])
val_videos = set(val_df['original_path'])
test_videos = set(test_df['original_path'])

assert len(train_videos.intersection(val_videos)) == 0, 'Leakage between train and val!'
assert len(train_videos.intersection(test_videos)) == 0, 'Leakage between train and test!'
assert len(val_videos.intersection(test_videos)) == 0, 'Leakage between val and test!'

print('No data leakage verified successfully!')


No data leakage verified successfully!


In [9]:
# Log summary stats to W&B
wb_manager = TrainingWandbManager(config, 'verify_metadata')
with wb_manager:
    stats = {
        'metadata/total_samples': len(tudat_df),
        'metadata/train_samples': len(train_df),
        'metadata/val_samples': len(val_df),
        'metadata/test_samples': len(test_df),
        'metadata/num_classes': len(class_counts),
    }
    for cls_name, count in class_counts.items():
        stats[f'metadata/class_count_{cls_name}'] = int(count)
    wb_manager.log_stats(stats)
    print('Logged metadata stats to W&B')


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Amita nagar\_netrc.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Logged metadata stats to W&B


metadata/class_count_accident,▁
metadata/class_count_challenging,▁
metadata/class_count_non-accident,▁
metadata/num_classes,▁
metadata/test_samples,▁
metadata/total_samples,▁
metadata/train_samples,▁
metadata/val_samples,▁
metadata/class_count_accident,44
metadata/class_count_challenging,17
metadata/class_count_non-accident,50
